# Phase 7: Clinical Decision-Support RAG System

This phase builds a **Retrieval-Augmented Generation (RAG)** system that turns a raw
model prediction into a structured **Diagnostic Support Report**, grounded in
verified dermatology sources (DermNet NZ) plus project-authored clinical summaries.

Pipeline:
1. **Knowledge base** -- per-class clinical descriptions, risk factors, and
   recommended actions, scraped from DermNet NZ with curated fallback content,
   saved to `results/rag/knowledge_base.json`.
2. **Vector store** -- chunks (200 words, 50-word overlap) embedded with
   `all-MiniLM-L6-v2` and indexed with FAISS (`IndexFlatL2`), saved to
   `results/rag/faiss_index.bin`.
3. **Retrieval pipeline** -- combines an exact class-match lookup with semantic
   top-k search, deduplicated into a `RetrievedContext`.
4. **Report generator** -- produces a structured "DIAGNOSTIC SUPPORT REPORT" with
   hallucination-prevention rules (source citations, confidence disclaimers,
   urgent-care warnings).
5. **`predict_and_explain`** -- combines the Phase 4 model, Phase 6 Grad-CAM++,
   and the medical knowledge base into one end-to-end function.

**Important**: This system is a research/educational prototype. It is **not** a
diagnostic device and must never be used as a substitute for professional
medical evaluation.

In [ ]:
import json
import os
import time
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import faiss
import requests
from bs4 import BeautifulSoup
from PIL import Image
from sentence_transformers import SentenceTransformer

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import efficientnet_b4

from config_loader import load_config, resolve_image_path

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR    = Path(r'c:\graduation project')
MODELS_DIR  = BASE_DIR / 'models' / 'checkpoints'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
RAG_DIR     = BASE_DIR / 'results' / 'rag'
RAG_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config(BASE_DIR)
CLASSES     = cfg['classes']
NUM_CLASSES = cfg['num_classes']
NORM_MEAN   = cfg['norm_mean']
NORM_STD    = cfg['norm_std']
IMG_SIZE    = cfg['img_size']
MALIGNANT_CLASSES = ['melanoma', 'basal cell carcinoma', 'squamous cell carcinoma']

print(f"Device: {DEVICE}")
print(f"Classes ({NUM_CLASSES}): {CLASSES}")

## 1 · Model + Grad-CAM++ (from Phases 4 & 6)

Reload the trained `SkinCancerModel` and the Grad-CAM++ implementation from Phase 6,
so this notebook can run `predict_and_explain` end-to-end without depending on the
other notebooks being open.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SkinCancerModel(nn.Module):
    IN_FEATURES = 1792

    def __init__(self, num_classes: int = 8):
        super().__init__()
        base = efficientnet_b4(weights=None)
        self.features = base.features
        self.se       = SEBlock(self.IN_FEATURES, reduction=16)
        self.avgpool  = base.avgpool
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.IN_FEATURES),
            nn.Linear(self.IN_FEATURES, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = SkinCancerModel(num_classes=NUM_CLASSES).to(DEVICE)
state_dict = torch.load(MODELS_DIR / 'final_model.pth', map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
print("Final model loaded ✓")

In [ ]:
class GradCAMPlusPlus:
    """Grad-CAM++ via forward/backward hooks on a target conv layer."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor: torch.Tensor, class_idx: int, percentile_clip: float = 90):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[:, class_idx]
        score.backward(retain_graph=True)

        grads = self.gradients
        acts  = self.activations

        grads2 = grads ** 2
        grads3 = grads2 * grads
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        eps = 1e-8
        alpha = grads2 / (2 * grads2 + sum_acts * grads3 + eps)
        alpha = torch.where(grads != 0, alpha, torch.zeros_like(alpha))
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()

        cap = np.percentile(cam, percentile_clip)
        if cap > 0:
            cam = np.minimum(cam, cap)

        cam_t = torch.from_numpy(cam).float().unsqueeze(0).unsqueeze(0)
        cam_t = F.interpolate(cam_t, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam_t.squeeze().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, output.softmax(dim=1).detach().cpu().numpy()[0]


gradcam = GradCAMPlusPlus(model, model.features[-1])

inference_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


def load_image_tensor(path: str):
    image = Image.open(path).convert('RGB')
    image = image.resize((IMG_SIZE, IMG_SIZE))
    tensor = inference_transform(image).unsqueeze(0).to(DEVICE)
    return image, tensor


def overlay_heatmap(image_pil: Image.Image, cam: np.ndarray, alpha: float = 0.45):
    image_np = np.asarray(image_pil).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * image_np
    return np.clip(overlay, 0, 1)


print("Model + Grad-CAM++ ready ✓")

## 2 · Knowledge Base Construction

For each of the 8 classes, build a knowledge-base entry containing:

- **`source`** -- where the content came from (DermNet NZ scrape or a
  project-authored curated summary).
- **`source_url`** -- the DermNet NZ topic page (for attribution / "Important
  Sources" in the report).
- **`source_date`** -- the date the content was scraped/written.
- **`content`** -- the clinical text itself.
- **`urgency`** -- `URGENT` (malignant: melanoma, basal cell carcinoma,
  squamous cell carcinoma), `MONITOR` (precancerous: actinic keratosis), or
  `ROUTINE` (benign: dermatofibroma, nevus, pigmented benign keratosis,
  vascular lesion).

**Strategy: attempt live scraping, fall back to curated.** We try to scrape the
main article body from each class's DermNet NZ page. Whether or not scraping
succeeds, we *always* also add a project-authored curated summary (clinical
description, risk factors, ABCDE relevance where applicable, and recommended
clinical action) -- this guarantees every class has consistent coverage of the
fields the report generator needs, even if a page layout changes or a request
fails. For melanoma we add one extra short, high-signal entry describing the
ABCDE "dark irregular lesion" warning signs, which improves semantic retrieval
for visual-description queries (verified below).

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
TODAY = date.today().isoformat()

# DermNet NZ source pages per class
DERMNET_URLS = {
    'actinic keratosis': 'https://dermnetnz.org/topics/actinic-keratosis',
    'basal cell carcinoma': 'https://dermnetnz.org/topics/basal-cell-carcinoma',
    'dermatofibroma': 'https://dermnetnz.org/topics/dermatofibroma',
    'melanoma': 'https://dermnetnz.org/topics/melanoma',
    'nevus': 'https://dermnetnz.org/topics/melanocytic-naevus',
    'pigmented benign keratosis': 'https://dermnetnz.org/topics/seborrhoeic-keratosis',
    'squamous cell carcinoma': 'https://dermnetnz.org/topics/cutaneous-squamous-cell-carcinoma',
    'vascular lesion': 'https://dermnetnz.org/topics/cherry-angioma',
}

# Urgency levels per class (clinical triage tier)
URGENCY = {
    'actinic keratosis': 'MONITOR',
    'basal cell carcinoma': 'URGENT',
    'dermatofibroma': 'ROUTINE',
    'melanoma': 'URGENT',
    'nevus': 'ROUTINE',
    'pigmented benign keratosis': 'ROUTINE',
    'squamous cell carcinoma': 'URGENT',
    'vascular lesion': 'ROUTINE',
}

URGENCY_ACTIONS = {
    'URGENT': (
        "This finding is associated with a malignant or potentially serious "
        "lesion. Prompt evaluation (within days) by a dermatologist is "
        "recommended, including biopsy if indicated."
    ),
    'MONITOR': (
        "This finding may represent a precancerous lesion. A dermatologist "
        "should evaluate it; early treatment can prevent progression to "
        "skin cancer."
    ),
    'ROUTINE': (
        "This finding is typically benign. Routine self-monitoring is "
        "sufficient. Consult a dermatologist if the lesion changes in size, "
        "shape, color, or becomes symptomatic."
    ),
}

print("Source URLs, urgency levels, and recommended actions defined ✓")

In [ ]:
# Curated fallback / supplementary summaries written for this project.
# Each gives: clinical description, ABCDE/diagnostic relevance, risk
# factors, and the recommended clinical action for the predicted class.
CURATED = {
    'actinic keratosis': (
        "Actinic keratosis is a precancerous, scaly or crusty patch of skin caused by "
        "cumulative sun (UV) damage. It typically appears as a rough, dry, pink-to-brown "
        "spot on chronically sun-exposed areas such as the face, ears, scalp, forearms and "
        "hands. Lesions feel rougher than they look and may be easier to feel than see. "
        "A small proportion of untreated actinic keratoses can progress to squamous cell "
        "carcinoma, so they are considered an early warning sign rather than a benign finding. "
        "Risk factors include fair skin, older age, chronic outdoor sun exposure, and "
        "immunosuppression. Recommended clinical action: a dermatologist should examine any "
        "new or changing scaly patch; treatment options include cryotherapy, topical "
        "field-directed therapy, photodynamic therapy, or curettage. Lesions that become "
        "thickened, tender, or ulcerated should be biopsied to exclude squamous cell carcinoma."
    ),
    'basal cell carcinoma': (
        "Basal cell carcinoma (BCC) is the most common form of skin cancer. It usually "
        "presents as a slow-growing, pearly or translucent bump, a flat scar-like lesion, "
        "or a sore that bleeds and does not heal, often with visible small blood vessels "
        "(telangiectasia) on its surface. BCC most often affects sun-exposed skin of the "
        "head and neck. It rarely spreads (metastasizes) to other parts of the body, but it "
        "can grow locally and damage surrounding skin, cartilage and bone if left untreated. "
        "Risk factors include cumulative UV exposure, fair skin, history of sunburns, older "
        "age, and a personal or family history of skin cancer. Recommended clinical action: "
        "any persistent non-healing sore or pearly growth should be evaluated promptly by a "
        "dermatologist; diagnosis is confirmed by biopsy and treatment is typically surgical "
        "excision or Mohs micrographic surgery."
    ),
    'dermatofibroma': (
        "Dermatofibroma is a common, benign, firm nodule found in the skin, most often on "
        "the legs. It is thought to arise from a reaction of skin tissue, sometimes after a "
        "minor injury such as an insect bite or a small cut. Dermatofibromas are usually "
        "brown, pink or reddish, firm to touch, and characteristically dimple inward when "
        "pinched from the sides (the 'dimple sign'). They are harmless and typically remain "
        "stable in size over years. Risk factors include minor trauma and being female, "
        "though many cases have no identifiable cause. Recommended clinical action: routine "
        "monitoring is generally sufficient; no treatment is required unless the lesion is "
        "symptomatic, repeatedly irritated, or its diagnosis is uncertain, in which case a "
        "dermatologist may biopsy or excise it to confirm the diagnosis."
    ),
    'melanoma': (
        "Melanoma is a malignant tumor of pigment-producing cells (melanocytes) and is the "
        "most dangerous form of skin cancer because of its potential to spread (metastasize) "
        "to other organs. It can develop within an existing mole or appear as a new "
        "pigmented lesion. The ABCDE criteria are used to flag suspicious lesions: "
        "Asymmetry (one half unlike the other), Border irregularity (uneven, notched or "
        "blurred edges), Color variation (multiple shades of brown, black, red, white or "
        "blue within one lesion), Diameter greater than 6mm, and Evolving (any change in "
        "size, shape, color, or new symptoms such as itching or bleeding). Risk factors "
        "include history of sunburns and intense intermittent UV exposure, fair skin, many "
        "or atypical moles, family history of melanoma, and immunosuppression. Recommended "
        "clinical action: any lesion meeting ABCDE criteria or showing rapid change requires "
        "urgent dermatologist referral and biopsy for histological diagnosis and staging. "
        "Early detection dramatically improves prognosis."
    ),
    'nevus': (
        "A melanocytic nevus (commonly called a mole) is a very common benign growth made "
        "of melanocytes. Typical nevi are small (usually under 6mm), symmetric, with a "
        "regular, well-defined border and a single, uniform shade of brown, tan or skin "
        "color, and they remain stable over time. Most adults have multiple nevi, and the "
        "number tends to increase with sun exposure during childhood and adolescence. The "
        "ABCDE criteria (Asymmetry, Border, Color, Diameter, Evolving) are used to "
        "distinguish typical nevi from melanoma; a benign nevus generally fails all five "
        "criteria. Risk factors for developing many nevi include fair skin, sun exposure, "
        "and genetic factors. Recommended clinical action: routine self-monitoring and "
        "periodic skin checks are sufficient for stable, typical-appearing nevi; any nevus "
        "that becomes asymmetric, develops irregular borders or color variation, enlarges, "
        "or starts to itch or bleed should be reviewed by a dermatologist."
    ),
    'pigmented benign keratosis': (
        "Pigmented benign keratosis covers a group of common, harmless, pigmented skin "
        "growths, the most frequent being seborrhoeic keratosis, along with solar lentigo "
        "and lichen-planus-like keratosis. Seborrhoeic keratoses appear as well-demarcated, "
        "'stuck-on' looking plaques with a warty or waxy surface, ranging in color from "
        "light tan to dark brown or black, and they become more numerous with age. Despite "
        "sometimes resembling melanoma due to their pigmentation and irregular outline, they "
        "are entirely benign and do not turn cancerous. Risk factors are mainly increasing "
        "age and genetic predisposition. Recommended clinical action: no treatment is "
        "necessary for typical lesions; removal (cryotherapy, curettage, or shave excision) "
        "may be offered if a lesion is irritated, catches on clothing, or for cosmetic "
        "reasons. A dermatologist should review any lesion where the diagnosis is uncertain "
        "or that has recently changed, to exclude melanoma."
    ),
    'squamous cell carcinoma': (
        "Squamous cell carcinoma (SCC) is the second most common form of skin cancer, "
        "arising from keratinocytes in the outer layer of skin. It typically presents as a "
        "scaly, crusted, red patch or a firm, dome-shaped nodule that may ulcerate or bleed, "
        "and it can develop from a pre-existing actinic keratosis. Unlike basal cell "
        "carcinoma, SCC carries a meaningful risk of spreading to lymph nodes and other "
        "organs if not treated, particularly on the lips, ears, and in immunosuppressed "
        "patients. Risk factors include cumulative UV exposure, fair skin, chronic wounds or "
        "scars, long-standing inflammation, and immunosuppression (e.g. organ transplant "
        "recipients). Recommended clinical action: any persistent scaly, ulcerated or "
        "rapidly growing lesion should be promptly biopsied; standard treatment is surgical "
        "excision, with Mohs surgery for high-risk sites, and closer follow-up is warranted "
        "given the metastatic potential."
    ),
    'vascular lesion': (
        "Vascular lesions of the skin are a broad group of benign growths formed from blood "
        "vessels, including cherry angiomas, angiokeratomas, pyogenic granulomas, and "
        "hemangiomas. The most common, cherry angiomas, appear as small, smooth, bright red "
        "to purple domed papules that increase in number with age and are essentially "
        "universal in older adults. Pyogenic granulomas grow rapidly, bleed easily, and "
        "often follow minor trauma. These lesions are generally harmless and do not become "
        "cancerous. Risk factors include increasing age (cherry angiomas), pregnancy and "
        "certain medications (for some vascular growths), and minor trauma (pyogenic "
        "granuloma). Recommended clinical action: routine monitoring is sufficient for "
        "stable, typical vascular lesions; treatment (laser, electrocautery, or excision) is "
        "optional and usually pursued for bleeding, irritation, or cosmetic concerns. Any "
        "vascular-appearing lesion that is rapidly enlarging, dark/blue-black, or "
        "diagnostically uncertain should be reviewed by a dermatologist to exclude other "
        "pigmented or malignant lesions."
    ),
}

# Extra short, high-signal entries to improve semantic retrieval for
# common visual-description queries (e.g. "dark irregular lesion").
EXTRA_ENTRIES = {
    'melanoma': [
        "A dark, irregularly shaped or irregularly colored skin lesion is a classic "
        "warning sign of melanoma. Suspicious features include a dark brown, black, "
        "or multi-colored lesion with an asymmetric shape, an uneven, notched or "
        "blurred border, and a diameter larger than 6mm. A dark irregular lesion that "
        "is new or changing should be treated as a possible melanoma until proven "
        "otherwise and referred for urgent dermatologist assessment and biopsy."
    ],
}

print(f"Curated summaries for {len(CURATED)} classes defined ✓")

In [ ]:
MAX_SCRAPED_CHARS = 6000


def extract_dermnet_content(url: str) -> str | None:
    """Fetch a DermNet NZ topic page and return its main article text, or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"  fetch failed: {e!r}")
        return None

    soup = BeautifulSoup(resp.text, 'html.parser')
    divs = soup.find_all('div', class_=lambda c: c and 'text-block' in c)

    candidates = []
    for d in divs:
        txt = d.get_text(separator=' ', strip=True)
        low = txt.lower()
        if low.startswith('advertisement') or low.startswith('references'):
            continue
        candidates.append(txt)

    if not candidates:
        return None

    main = max(candidates, key=len)
    if len(main) < 500:
        return None

    return main[:MAX_SCRAPED_CHARS]


kb_entries = []
for cls, url in DERMNET_URLS.items():
    urgency = URGENCY[cls]
    print(f"Scraping {cls!r} from {url} ...")
    scraped = extract_dermnet_content(url)

    if scraped:
        print(f"  scraped {len(scraped)} chars")
        kb_entries.append({
            'class': cls,
            'source': 'DermNet NZ',
            'source_url': url,
            'source_date': TODAY,
            'content': scraped,
            'urgency': urgency,
        })
    else:
        print("  scraping failed, will rely on curated content only")

    # Always include the curated summary too, for consistent coverage of
    # risk factors / recommended action / urgency.
    kb_entries.append({
        'class': cls,
        'source': 'Curated summary (project-authored, based on DermNet NZ / AAD / WHO IARC)',
        'source_url': url,
        'source_date': TODAY,
        'content': CURATED[cls],
        'urgency': urgency,
    })

    for extra in EXTRA_ENTRIES.get(cls, []):
        kb_entries.append({
            'class': cls,
            'source': 'Curated clinical warning signs (project-authored)',
            'source_url': url,
            'source_date': TODAY,
            'content': extra,
            'urgency': urgency,
        })

    time.sleep(0.5)

with open(RAG_DIR / 'knowledge_base.json', 'w', encoding='utf-8') as f:
    json.dump(kb_entries, f, indent=2, ensure_ascii=False)

print(f"\nWrote {len(kb_entries)} entries to {RAG_DIR / 'knowledge_base.json'}")

## 3 · Embedding & Vector Store (FAISS)

Each knowledge-base entry is split into **200-word chunks with 50-word overlap**.
Chunks are embedded with `all-MiniLM-L6-v2` (384-dim, L2-normalized so cosine
similarity == dot product) and indexed with FAISS `IndexFlatL2` for exact
nearest-neighbor search over this small corpus.

We verify the index with the query *"dark irregular lesion"*, which should
retrieve melanoma content as the top match -- the canonical ABCDE warning sign.

In [ ]:
CHUNK_WORDS = 200
CHUNK_OVERLAP = 50


def chunk_text(text: str, chunk_words: int = CHUNK_WORDS, overlap: int = CHUNK_OVERLAP):
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    step = chunk_words - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_words]
        if not piece:
            break
        chunks.append(' '.join(piece))
        if start + chunk_words >= len(words):
            break
    return chunks


chunk_records = []
for entry in kb_entries:
    for chunk in chunk_text(entry['content']):
        chunk_records.append({
            'chunk_id': len(chunk_records),
            'class': entry['class'],
            'source': entry['source'],
            'source_url': entry['source_url'],
            'source_date': entry['source_date'],
            'urgency': entry['urgency'],
            'text': chunk,
        })

print(f"Built {len(chunk_records)} chunks from {len(kb_entries)} KB entries")

embedder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE.type if DEVICE.type == 'cuda' else 'cpu')

chunk_texts = [c['text'] for c in chunk_records]
chunk_embeddings = embedder.encode(
    chunk_texts, convert_to_numpy=True, normalize_embeddings=True,
    batch_size=32, show_progress_bar=True,
).astype('float32')

print("Embeddings shape:", chunk_embeddings.shape)

faiss_index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)
print("FAISS index size:", faiss_index.ntotal)

faiss.write_index(faiss_index, str(RAG_DIR / 'faiss_index.bin'))
with open(RAG_DIR / 'chunk_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

print(f"Saved index to {RAG_DIR / 'faiss_index.bin'}")
print(f"Saved metadata to {RAG_DIR / 'chunk_metadata.json'}")

In [ ]:
def semantic_search(query: str, k: int = 5):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    distances, indices = faiss_index.search(q_emb, k)
    return [(chunk_records[i], float(d)) for i, d in zip(indices[0], distances[0])]


print('Query: "dark irregular lesion" -> top 5 chunks:')
for rank, (rec, dist) in enumerate(semantic_search("dark irregular lesion", k=5)):
    print(f"  {rank+1}. class={rec['class']:28s} dist={dist:.4f}  source={rec['source'][:40]}")
    print(f"     {rec['text'][:110]}...")

top_class = semantic_search("dark irregular lesion", k=1)[0][0]['class']
assert top_class == 'melanoma', f"Expected melanoma as top match, got {top_class!r}"
print("\nVerification passed: 'dark irregular lesion' retrieves melanoma content ✓")

## 4 · Retrieval Pipeline

`retrieve_for_class()` builds a `RetrievedContext` for a predicted class by
combining:

1. **Exact class match** -- every chunk whose `class` field equals the
   predicted class (guarantees the report always has the curated description,
   risk factors, and recommended action for that class).
2. **Semantic top-k** -- the `k` chunks (across all classes) most similar to
   the predicted class name, which can surface related differential-diagnosis
   context (e.g. melanoma vs. nevus vs. pigmented benign keratosis).
3. **Deduplication** -- chunks are deduplicated by `chunk_id`, and sources are
   deduplicated into a single attribution list for the report.

In [ ]:
@dataclass
class RetrievedContext:
    predicted_class: str
    confidence: float
    chunks: list = field(default_factory=list)
    sources: list = field(default_factory=list)


def retrieve_for_class(predicted_class: str, confidence: float, k: int = 3) -> RetrievedContext:
    # 1. Exact class match
    class_chunks = [c for c in chunk_records if c['class'] == predicted_class]

    # 2. Semantic top-k (query = class name, catches related differential context)
    semantic_chunks = [c for c, _ in semantic_search(predicted_class, k=k)]

    # 3. Dedup by chunk_id, exact-class chunks first
    seen_ids = set()
    combined = []
    for c in class_chunks + semantic_chunks:
        if c['chunk_id'] not in seen_ids:
            seen_ids.add(c['chunk_id'])
            combined.append(c)

    # Dedup sources into an attribution list
    seen_src = set()
    sources = []
    for c in combined:
        key = (c['source'], c['source_url'])
        if key not in seen_src:
            seen_src.add(key)
            sources.append(f"{c['source']} — {c['source_url']} (accessed {c['source_date']})")

    return RetrievedContext(predicted_class=predicted_class, confidence=confidence,
                             chunks=combined, sources=sources)


# Quick smoke test
_ctx = retrieve_for_class('melanoma', confidence=0.87)
print(f"retrieve_for_class('melanoma', 0.87) -> {len(_ctx.chunks)} chunks, {len(_ctx.sources)} sources")
for s in _ctx.sources:
    print("  -", s)

## 5 · Diagnostic Support Report Generator

`generate_report()` produces a structured plain-text report with the following
sections: **Predicted Condition**, **Confidence**, **Urgency**, **What Is
This?**, **Visual Characteristics (Grad-CAM++)**, **Clinical Recommendation**,
**Top Differential Diagnoses**, **Important Sources**, and a closing
**disclaimer**.

**Hallucination-prevention rules** baked into the generator:

- Every report cites the KB sources it drew from (`Important Sources`).
- If model confidence is below **70%**, a low-confidence disclaimer is added
  recommending professional review.
- If the predicted class is **melanoma** with confidence above **80%**, an
  explicit **"SEEK IMMEDIATE MEDICAL CARE"** warning is added.
- If no knowledge-base content is retrieved for a class (offline / KB
  unavailable), the report falls back to a generic safety message instead of
  fabricating clinical content.

In [ ]:
def describe_gradcam(cam: np.ndarray, threshold: float = 0.5) -> str:
    """Turn a Grad-CAM++ heatmap into a short natural-language description."""
    h, w = cam.shape
    mask = cam >= threshold
    if not mask.any():
        return ("The model's attention map (Grad-CAM++) was diffuse, with no single "
                "dominant focus region. Compare the heatmap overlay to the lesion "
                "location to assess whether the model attended to the lesion.")

    ys, xs = np.nonzero(mask)
    cy, cx = ys.mean() / h, xs.mean() / w
    coverage = mask.mean()

    vert = 'upper' if cy < 0.4 else ('lower' if cy > 0.6 else 'central')
    horiz = 'left' if cx < 0.4 else ('right' if cx > 0.6 else 'central')
    if vert == 'central' and horiz == 'central':
        position = 'the central region'
    elif vert == 'central':
        position = f'the {horiz} side'
    elif horiz == 'central':
        position = f'the {vert} part'
    else:
        position = f'the {vert}-{horiz} region'

    extent = 'a tightly focused area' if coverage < 0.20 else (
        'a moderate area' if coverage < 0.45 else 'a broad area')

    return (f"The model's attention (Grad-CAM++) was concentrated on {extent} in "
            f"{position} of the image, covering approximately {coverage * 100:.0f}% "
            f"of the frame at high activation. Compare this overlay to the lesion's "
            f"location in the original image -- attention concentrated on the lesion "
            f"itself supports the prediction, while attention on skin, hair, or "
            f"borders would indicate the prediction may be unreliable.")


print("describe_gradcam() defined ✓")

In [ ]:
URGENCY_LABELS = {
    'URGENT': 'URGENT - prompt dermatologist evaluation recommended',
    'MONITOR': 'MONITOR - dermatologist review recommended (not an emergency)',
    'ROUTINE': 'ROUTINE - routine self-monitoring is sufficient',
}

DISCLAIMER = (
    "DISCLAIMER: This report was generated automatically by an AI research "
    "prototype for an educational graduation project. It is NOT a medical "
    "diagnosis and has NOT been validated for clinical use. All findings must "
    "be reviewed by a qualified dermatologist or physician. Do not delay "
    "seeking medical care based on this output."
)


def generate_report(predicted_class: str, confidence: float, top3: list,
                     gradcam_description: str, context: RetrievedContext) -> str:
    lines = []
    lines.append("=" * 70)
    lines.append("DIAGNOSTIC SUPPORT REPORT")
    lines.append("=" * 70)

    lines.append(f"\nPredicted Condition: {predicted_class.title()}")
    lines.append(f"Confidence: {confidence * 100:.1f}%")

    class_chunks = [c for c in context.chunks if c['class'] == predicted_class]
    urgency = class_chunks[0]['urgency'] if class_chunks else 'ROUTINE'
    lines.append(f"Urgency Level: {URGENCY_LABELS.get(urgency, urgency)}")

    lines.append("\n--- What Is This? ---")
    desc_chunk = next((c for c in class_chunks if c['source'].startswith('Curated summary')), None)
    if desc_chunk is None:
        desc_chunk = class_chunks[0] if class_chunks else None
    if desc_chunk:
        lines.append(desc_chunk['text'])
    else:
        lines.append(
            "No knowledge-base content is available for this class (offline "
            "fallback). Please consult a dermatologist for information about "
            "this condition."
        )

    lines.append("\n--- Visual Characteristics (Grad-CAM++) ---")
    lines.append(gradcam_description)

    lines.append("\n--- Clinical Recommendation ---")
    lines.append(URGENCY_ACTIONS.get(urgency, URGENCY_ACTIONS['ROUTINE']))

    lines.append("\n--- Top Differential Diagnoses ---")
    for cls, p in top3:
        lines.append(f"  - {cls.title()}: {p * 100:.1f}%")

    lines.append("\n--- Important Sources ---")
    if context.sources:
        for s in context.sources:
            lines.append(f"  - {s}")
    else:
        lines.append("  - No sources retrieved (offline fallback).")

    # --- Hallucination-prevention rules ---
    if confidence < 0.70:
        lines.append(
            "\n[NOTE] Model confidence is below 70%. This prediction is "
            "uncertain and should not be relied upon -- please consult a "
            "dermatologist for an accurate diagnosis."
        )

    if predicted_class == 'melanoma' and confidence > 0.80:
        lines.append(
            "\n[WARNING] High-confidence melanoma prediction. SEEK IMMEDIATE "
            "MEDICAL CARE from a dermatologist or healthcare provider for "
            "evaluation and biopsy."
        )

    lines.append("\n" + "=" * 70)
    lines.append(DISCLAIMER)
    lines.append("=" * 70)

    return "\n".join(lines)


print("generate_report() defined ✓")

## 6 · `predict_and_explain` -- End-to-End Integration

This is the main entry point of the clinical RAG system. Given an image path,
it returns:

- `prediction` -- the predicted class name
- `confidence` -- softmax probability of the predicted class
- `top3` -- the top-3 `(class, probability)` differential diagnoses
- `gradcam_image` -- the Grad-CAM++ overlay (RGB array, 0-1 range)
- `medical_report` -- the structured Diagnostic Support Report (string)

In [ ]:
def predict_and_explain(image_path: str) -> dict:
    image_pil, tensor = load_image_tensor(image_path)

    with torch.no_grad():
        logits = model(tensor)
        proba = logits.softmax(dim=1).cpu().numpy()[0]

    top3_idx = np.argsort(-proba)[:3]
    top3 = [(CLASSES[i], float(proba[i])) for i in top3_idx]
    pred_idx = int(top3_idx[0])
    pred_class = CLASSES[pred_idx]
    confidence = float(proba[pred_idx])

    cam, _ = gradcam.generate(tensor, pred_idx)
    overlay = overlay_heatmap(image_pil, cam)
    gradcam_description = describe_gradcam(cam)

    context = retrieve_for_class(pred_class, confidence)
    report = generate_report(pred_class, confidence, top3, gradcam_description, context)

    return {
        'prediction': pred_class,
        'confidence': confidence,
        'top3': top3,
        'gradcam_image': overlay,
        'medical_report': report,
        'original_image': image_pil,
    }


print("predict_and_explain() defined ✓")

## 7 · Demo

Run `predict_and_explain` on three representative test-set cases:

1. A **high-confidence, correctly-classified melanoma** -- triggers the
   "SEEK IMMEDIATE MEDICAL CARE" warning.
2. A **high-confidence, correctly-classified nevus** -- a routine, benign
   result.
3. A **low-confidence case** (model confidence < 70%) -- triggers the
   low-confidence disclaimer.

In [ ]:
pred_df = pd.read_csv(METRICS_DIR / 'test_predictions.csv')
proba_cols = [f"proba_{c.replace(' ', '_')}" for c in CLASSES]
pred_df['pred_proba'] = pred_df[proba_cols].max(axis=1)

melanoma_tp = pred_df[(pred_df['label_str'] == 'melanoma') & (pred_df['pred_str'] == 'melanoma')] \
    .sort_values('proba_melanoma', ascending=False).iloc[0]

nevus_tp = pred_df[(pred_df['label_str'] == 'nevus') & (pred_df['pred_str'] == 'nevus')] \
    .sort_values('proba_nevus', ascending=False).iloc[0]

low_conf = pred_df[pred_df['pred_proba'] < 0.70].sort_values('pred_proba', ascending=False).iloc[0]

demo_cases = {
    'High-confidence melanoma (TP)': melanoma_tp,
    'High-confidence nevus (TP)': nevus_tp,
    'Low-confidence case': low_conf,
}

for name, row in demo_cases.items():
    print(f"{name}: true={row['label_str']!r}, pred={row['pred_str']!r}, "
          f"confidence={row['pred_proba']:.3f}")

In [ ]:
for name, row in demo_cases.items():
    path = resolve_image_path(row, cfg)
    result = predict_and_explain(path)

    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    axes[0].imshow(result['original_image'])
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(result['gradcam_image'])
    axes[1].set_title('Grad-CAM++ overlay')
    axes[1].axis('off')
    fig.suptitle(f"{name}  (true label: {row['label_str']})")
    plt.tight_layout()
    plt.show()

    print(result['medical_report'])
    print("\n")

### Safety-rule check (synthetic)

The test set never produces a melanoma prediction above 80% confidence (max
observed: ~60%), so the "high-confidence melanoma" safety warning never fires
on real predictions here. To confirm the rule itself is correct, generate a
report from a **synthetic** high-confidence melanoma scenario (same retrieved
context, hypothetical confidence) and check that the urgent-care warning
appears.


In [ ]:
_ctx = retrieve_for_class("melanoma", confidence=0.85)
_synthetic_top3 = [("melanoma", 0.85), ("pigmented benign keratosis", 0.10), ("nevus", 0.05)]
_synthetic_report = generate_report("melanoma", 0.85, _synthetic_top3,
                                      "(synthetic scenario -- no Grad-CAM computed)", _ctx)

assert "SEEK IMMEDIATE MEDICAL CARE" in _synthetic_report, "Urgent-care warning did not fire"
assert "below 70%" not in _synthetic_report, "Low-confidence disclaimer should not fire at 85%"
print("Safety-rule check passed: >80% melanoma confidence triggers urgent-care warning ✓")
print()
print(_synthetic_report)


## Phase 7 Complete

**Outputs**:
- `results/rag/knowledge_base.json` -- per-class clinical descriptions, risk factors, recommended actions, and urgency levels (DermNet NZ + curated, with attribution).
- `results/rag/chunk_metadata.json` -- 200-word/50-word-overlap chunks with class, source, source_url, source_date, urgency.
- `results/rag/faiss_index.bin` -- FAISS `IndexFlatL2` over all-MiniLM-L6-v2 embeddings of the chunks.
- `predict_and_explain(image_path)` -- combines the Phase 4 model, Phase 6 Grad-CAM++, and this medical knowledge base into a structured Diagnostic Support Report.

**Safety design**:
- Every report cites its sources.
- Confidence < 70% triggers a low-confidence disclaimer.
- Melanoma predictions with confidence > 80% trigger an explicit "seek immediate care" warning.
- A standing disclaimer on every report states this is a research prototype, not a diagnostic device, and findings must be reviewed by a clinician.
